# M1 — Static Additive Multi-Loss
`L = 0.5·L_cw_wsft + 0.3·L_dec_only + 0.2·L_dec_ent`

Combines three complementary losses with fixed weights. This is the baseline for all other multi-objective methods.

In [ ]:
# Cell 1: Imports and setup
import os, sys, json, math, re
from typing import List, Dict, Any, Tuple

import torch
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer

sys.path.insert(0, os.getcwd())
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, EPOCHS, LR, BATCH_SIZE, GRAD_ACC, SEED,
    W_FORMAT, W_DECISION, W_EXPL, LAMBDA,
    STUDENTS, load_data, load_student,
    get_section_spans, in_any_span,
    compute_alpha, compute_mean_confidence,
    find_decision_span_chars, teacher_section_entropy_mean,
    build_prompt_text, tokenize_and_mask, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "m1_additive_multi_loss")
os.makedirs(OUT_DIR, exist_ok=True)

BETA_CW_WSFT = 0.5
BETA_DEC_ONLY = 0.3
BETA_DEC_ENT = 0.2

prompts, teacher_rows = load_data()
MEAN_C = compute_mean_confidence(teacher_rows)
print(f"Mean confidence: {MEAN_C:.6f}")
print(f"Output dir: {OUT_DIR}")

In [ ]:
# Cell 2: Dataset
class M1Dataset(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct, mean_c):
        self.rows, self.prompts, self.tokenizer = rows, prompts, tokenizer
        self.is_instruct, self.mean_c = is_instruct, mean_c

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        prompt = self.prompts[r["id"]]
        answer = r["teacher_text"]

        prompt_text = build_prompt_text(self.tokenizer, prompt, self.is_instruct)
        input_ids, offsets, labels, answer_start = tokenize_and_mask(
            self.tokenizer, prompt_text, answer
        )

        # WSFT weights
        wsft_weights = [0.0] * len(input_ids)
        spans = get_section_spans(answer)
        dec_spans = [(answer_start + s, answer_start + e) for s, e in spans["decision"]]
        expl_spans = [(answer_start + s, answer_start + e) for s, e in spans["explanation"]]

        for i, (s, e) in enumerate(offsets):
            if e <= s: continue
            if s >= answer_start:
                w = W_FORMAT
                if in_any_span(s, e, dec_spans): w = W_DECISION
                if in_any_span(s, e, expl_spans): w = W_EXPL
                wsft_weights[i] = float(w)

        active_w = [w for w in wsft_weights if w > 0]
        if active_w:
            mean_w = sum(active_w) / len(active_w)
            if mean_w > 1e-6:
                wsft_weights = [w / mean_w if w > 0 else 0.0 for w in wsft_weights]

        alpha = compute_alpha(r, self.mean_c)

        # Decision mask
        dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
        dec_span_chars = find_decision_span_chars(answer)
        if dec_span_chars:
            ds, de = dec_span_chars
            full_ds, full_de = answer_start + ds, answer_start + de
            for i, (s, e) in enumerate(offsets):
                if labels[i] != -100 and not (e <= full_ds or s >= full_de):
                    dec_mask[i] = True

        ent_teacher = teacher_section_entropy_mean(r, dec_span_chars)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "wsft_weights": torch.tensor(wsft_weights, dtype=torch.float),
            "alpha": torch.tensor(alpha, dtype=torch.float),
            "dec_mask": dec_mask,
            "ent_teacher": ent_teacher.float(),
        }

print("✅ Dataset class ready")

In [ ]:
# Cell 3: Trainer with combined loss
class M1Trainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        wsft_weights = inputs.pop("wsft_weights")
        alpha = inputs.pop("alpha")
        dec_mask = inputs.pop("dec_mask")
        ent_teacher = inputs.pop("ent_teacher")

        outputs = model(**inputs)
        logits = outputs.logits

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        shift_wsft_w = wsft_weights[:, 1:].contiguous()
        shift_dec_mask = dec_mask[:, 1:]

        vocab = shift_logits.size(-1)
        B = shift_logits.size(0)

        token_loss = torch.nn.CrossEntropyLoss(reduction="none")(
            shift_logits.view(-1, vocab), shift_labels.view(-1),
        ).view(shift_labels.size())

        active = (shift_labels != -100).float()

        # L_cw_wsft
        w = shift_wsft_w * active
        denom = active.sum(dim=1).clamp(min=1.0)
        per_sample = (token_loss * w).sum(dim=1) / denom
        L_cw_wsft = (per_sample * alpha).mean()

        # L_dec_only
        dec_active = shift_dec_mask.float() * active
        dec_denom = dec_active.sum().clamp(min=1.0)
        L_dec_only = (token_loss * dec_active).sum() / dec_denom

        # L_dec_ent
        probs = torch.softmax(shift_logits, dim=-1)
        ent_tok = -(probs * torch.log(probs + 1e-9)).sum(-1)
        ent_student = torch.zeros(B, device=logits.device)
        for b in range(B):
            m = shift_dec_mask[b]
            if m.any():
                ent_student[b] = ent_tok[b][m].mean()
        L_dec_ent = LAMBDA * ((ent_student - ent_teacher.to(logits.device)) ** 2).mean()

        loss = BETA_CW_WSFT * L_cw_wsft + BETA_DEC_ONLY * L_dec_only + BETA_DEC_ENT * L_dec_ent
        return (loss, outputs) if return_outputs else loss

print("✅ Trainer class ready")

In [ ]:
# Cell 4: Train all students
for model_name, cfg in STUDENTS.items():
    print(f"\n{'='*50} M1: {model_name} {'='*50}")

    out_path = os.path.join(OUT_DIR, model_name)
    os.makedirs(out_path, exist_ok=True)

    tokenizer, model = load_student(model_name, cfg)
    dataset = M1Dataset(teacher_rows, prompts, tokenizer, cfg["is_instruct"], MEAN_C)
    collator = FlexibleCollator(
        tokenizer,
        extra_1d_fields=["wsft_weights", "dec_mask"],
        extra_scalar_fields=["alpha", "ent_teacher"],
    )

    trainer = M1Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=out_path,
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACC,
            learning_rate=LR,
            logging_steps=25,
            save_strategy="no",
            bf16=True,
            seed=SEED,
            report_to="none",
            remove_unused_columns=False,
            dataloader_num_workers=2,
        ),
        train_dataset=dataset,
        data_collator=collator,
    )

    trainer.train()
    model.save_pretrained(out_path)
    tokenizer.save_pretrained(out_path)
    print(f"✅ Saved {model_name} → {out_path}")

    del model, tokenizer, trainer
    torch.cuda.empty_cache()

print("\n✅ M1 complete for all students.")